# NearQuery: the open notebook
Re-run the thesis numbers yourself. Everything here reads the committed snapshots in `data/metrics/` (derived from public NEAR mainnet logs; DefiLlama's own series are never stored). No accounts or keys.

Run it: `uv run --with jupyterlab --with matplotlib jupyter lab notebooks/nearquery.ipynb`

In [ ]:
from pathlib import Path
import duckdb, pandas as pd
import matplotlib.pyplot as plt
M = next(p for p in [Path('data/metrics'), Path('../data/metrics')] if p.exists())
con = duckdb.connect()
def q(sql): return con.execute(sql.replace('{M}', str(M))).df()
plt.rcParams.update({'figure.figsize': (9, 3.2), 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Volume and intents per day (public on-chain)

In [ ]:
v = pd.read_csv(M/'daily_volume.csv', parse_dates=['date']).iloc[:-1]  # newest day is partial
ax = v.set_index('date')['leg_sum_usd'].rolling(7).mean().plot(title='Leg-sum USD per day, 7-day mean')
v.tail(3)

## 2. The implied confidential share, and the 2026-09-01 step
One minus public leg-sum over DefiLlama's reported volume. A residual, not a measurement.

In [ ]:
w = pd.read_csv(M/'confidential_weekly.csv', parse_dates=['week_start'])
w.set_index('week_start')['implied_confidential_share'].mul(100).plot(title='Implied confidential share (%), weekly')
pd.read_csv(M/'confidential_break.csv', index_col=0)

## 3. Classify swaps by source chain and pair
Monthly user-side volume by the origin chain of the asset sold and of the asset bought.

In [ ]:
q("""
SELECT sold_chain || ' -> ' || bought_chain AS pair, round(sum(usd)/1e6) AS usd_millions, sum(swaps) AS swaps
FROM read_csv_auto('{M}/chain_pairs.csv') WHERE month >= '2026-08-01'
GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""")

## 4. Wash and bot screen
Two stated rules (50+ swaps in a day; round trips within a day). The SQL is `pipeline/sql/90_wash_screen.sql`.

In [ ]:
ws = pd.read_csv(M/'wash_screen.csv')
ws[['month','user_volume_usd','high_frequency_share','round_trip_share','kept_share']]

## 5. Fees per $1M against other cross-chain protocols
DefiLlama's 30-day fees over volume, as it defines each; a rough benchmark. Rule in `pipeline/peers.py`.

In [ ]:
pf = pd.read_csv(M/'peer_fees.csv')
pf.set_index('protocol')['fees_per_1m_usd'].sort_values().plot.barh(title='Fees per $1M of volume (USD)')
pf

## 6. What happens to NEAR demand at 10x volume?
Change the assumptions and recompute the buyback yield. Same formula as `pipeline/model.py`.

In [ ]:
obs = pd.read_csv(M/'model_observed.csv', index_col=0)['value']
rev, mcap = float(obs['annualized_revenue_usd']), float(obs['near_mcap_usd'])
def buyback(volume_x=1, take_x=1, price_x=1):
    annual = rev * volume_x * take_x
    return annual, annual / (mcap * price_x)
for x in (1, 5, 10):
    a, pct = buyback(x)
    print(f'{x:>2}x volume: ${a/1e6:,.1f}M a year = {pct:.2%} of market cap')